# Module 1 · Notebook 1: Autograd on single numbers

**Question:** how does a computer compute gradients automatically, and how do we *prove* they are correct?

Before NumPy arrays, we build the idea on plain scalars, where every step can be checked by hand.

| Step | What we do |
|---|---|
| 1 | What a gradient is, and the finite-difference "ground truth" |
| 2 | Chain rule by hand on a tiny expression |
| 3 | A `Value` class: each operation stores its *local* derivative |
| 4 | `backward()`: walk the graph in reverse (topological order) |
| 5 | Prove every operation correct with a gradient check |
| 6 | Two classic bugs, and how the gradient check catches them |

In [17]:
import sys, math
import numpy as np

print("python", sys.version.split()[0], "| numpy", np.__version__)

python 3.12.15 | numpy 2.5.3


## Step 1: What is a gradient?

The derivative $\frac{df}{dx}$ says: *if I nudge $x$ a tiny bit, how much does $f$ change?*

We can measure that directly by nudging. The **central difference**:

$$\frac{df}{dx} \approx \frac{f(x+h) - f(x-h)}{2h}$$

This is slow (2 function calls *per input*, so useless for a model with 1M parameters), but it needs no calculus. That makes it the perfect **referee** for checking our autograd later.

In [18]:
def f(x):
    return x**2 + 3*x          # by hand: df/dx = 2x + 3

def numerical_grad(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

x = 2.0
print("numerical:", numerical_grad(f, x))
print("analytic :", 2*x + 3)

numerical: 7.000000000090267
analytic : 7.0


## Step 2: Chain rule by hand

Take $L = (a \cdot b + c) \cdot a$. Split it into small steps:

$$e = a \cdot b \qquad d = e + c \qquad L = d \cdot a$$

Notice **$a$ is used twice**: once in $e$, once directly in $L$. So the gradient reaches $a$ along two paths, and the two contributions **add**:

$$\frac{\partial L}{\partial a} = \underbrace{d}_{\text{direct path}} + \underbrace{a \cdot 1 \cdot b}_{\text{path through } e} \qquad \frac{\partial L}{\partial b} = a \cdot a \qquad \frac{\partial L}{\partial c} = a$$

In [19]:
a, b, c = 2.0, -3.0, 10.0

def L_fn(a, b, c):
    e = a * b
    d = e + c
    return d * a

# hand-derived with the chain rule
e = a * b
d = e + c
dL_da = d + a * b     # two paths add up
dL_db = a * a
dL_dc = a

h = 1e-6
num_da = (L_fn(a + h, b, c) - L_fn(a - h, b, c)) / (2 * h)
num_db = (L_fn(a, b + h, c) - L_fn(a, b - h, c)) / (2 * h)
num_dc = (L_fn(a, b, c + h) - L_fn(a, b, c - h)) / (2 * h)

print("L =", L_fn(a, b, c))
print(f"dL/da  hand={dL_da:+.6f}  numerical={num_da:+.6f}")
print(f"dL/db  hand={dL_db:+.6f}  numerical={num_db:+.6f}")
print(f"dL/dc  hand={dL_dc:+.6f}  numerical={num_dc:+.6f}")

L = 8.0
dL/da  hand=-2.000000  numerical=-2.000000
dL/db  hand=+4.000000  numerical=+4.000000
dL/dc  hand=+2.000000  numerical=+2.000000


## Step 3: The key idea of autograd

Doing Step 2 by hand for a transformer is impossible. The trick:

1. **Each operation only knows its own local derivative.** Multiplication knows $\frac{\partial (xy)}{\partial x} = y$. It knows nothing about the rest of the model.
2. While computing forward, every result **remembers which values produced it** (this builds a graph).
3. Going backward, each node takes the gradient arriving from above, multiplies by its local derivative, and **adds** it into its inputs' `.grad`.

| Operation | Forward | Local derivative(s) |
|---|---|---|
| add | $x + y$ | $1,\ 1$ |
| mul | $x \cdot y$ | $y,\ x$ |
| pow | $x^n$ | $n x^{n-1}$ |
| exp | $e^x$ | $e^x$ |
| log | $\ln x$ | $1/x$ |
| tanh | $t = \tanh x$ | $1 - t^2$ |

Subtraction and division need **no new derivative code**: $x - y = x + (-1 \cdot y)$ and $x / y = x \cdot y^{-1}$.

In [20]:
class Value:
    # A single number that remembers how it was computed.

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0                  # dL/d(self), filled in by backward()
        self._prev = set(_children)      # the Values this one was computed from
        self._op = _op                   # for debugging / printing
        self._backward = lambda: None    # pushes out.grad into the children

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ---- primitive ops: each one defines its local derivative ----

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        assert isinstance(n, (int, float)), "only constant exponents"
        out = Value(self.data ** n, (self,), f"**{n}")
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    # ---- derived ops: built from the primitives, no new derivative code ----

    def __neg__(self):              return self * -1
    def __sub__(self, other):       return self + (-other)
    def __truediv__(self, other):   return self * other**-1
    def __radd__(self, other):      return self + other
    def __rmul__(self, other):      return self * other
    def __rsub__(self, other):      return other + (-self)
    def __rtruediv__(self, other):  return other * self**-1


# forward pass only: grads are still 0 because nobody has run backward yet
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
L = (a * b + c) * a
print(L, "| built by op:", L._op, "| from", len(L._prev), "inputs")

Value(data=8.0000, grad=0.0000) | built by op: * | from 2 inputs


## Step 4: `backward()`, and why order matters

A node may only push its gradient to its inputs **after its own gradient is complete**, i.e. after *every* node that uses it has already run.

That order is a **topological sort** of the graph, reversed:
- Build a list where each node comes *after* everything it depends on.
- Set $\frac{\partial L}{\partial L} = 1$.
- Walk the list backwards, calling each node's `_backward`.

In [21]:
def backward(self):
    order, visited = [], set()

    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            order.append(v)          # v goes in only after all its inputs

    build(self)
    self.grad = 1.0                  # dL/dL = 1
    for v in reversed(order):
        v._backward()

Value.backward = backward

# same expression as Step 2, now with autograd
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
L = (a * b + c) * a
L.backward()

print(L)
for name, v, hand in [("a", a, dL_da), ("b", b, dL_db), ("c", c, dL_dc)]:
    print(f"dL/d{name}: autograd={v.grad:+.6f}   hand={hand:+.6f}")

Value(data=8.0000, grad=1.0000)
dL/da: autograd=-2.000000   hand=-2.000000
dL/db: autograd=+4.000000   hand=+4.000000
dL/dc: autograd=+2.000000   hand=+2.000000


## Step 5: Prove every operation is correct

Matching one hand example is not proof. A **gradient check** compares autograd against finite differences for *every* operation, using the **relative error**:

$$\text{rel\_err} = \frac{|g_{\text{auto}} - g_{\text{num}}|}{|g_{\text{auto}}| + |g_{\text{num}}|}$$

Relative, not absolute, because a gradient of 1000 off by 0.001 is fine, but a gradient of 0.001 off by 0.001 is 100% wrong. In float64 with $h = 10^{-6}$, a correct op typically lands around $10^{-9}$ to $10^{-11}$.

In [22]:
def gradcheck(fn, xs, h=1e-6):
    # fn: takes a list of Values, returns one Value. xs: list of floats.
    vals = [Value(x) for x in xs]
    fn(vals).backward()

    worst = 0.0
    for i in range(len(xs)):
        plus  = list(xs); plus[i]  += h
        minus = list(xs); minus[i] -= h
        num = (fn([Value(x) for x in plus]).data - fn([Value(x) for x in minus]).data) / (2 * h)
        ana = vals[i].grad
        err = abs(ana - num) / max(abs(ana) + abs(num), 1e-12)
        worst = max(worst, err)
    return worst

tests = {
    "add":             lambda v: v[0] + v[1],
    "mul":             lambda v: v[0] * v[1],
    "sub":             lambda v: v[0] - v[1],
    "div":             lambda v: v[0] / v[1],
    "pow":             lambda v: v[0] ** 3,
    "exp":             lambda v: v[0].exp(),
    "log":             lambda v: v[0].log(),
    "tanh":            lambda v: v[0].tanh(),
    "reuse: x*x + x":  lambda v: v[0] * v[0] + v[0],
    "mixed":           lambda v: ((v[0] * v[1]).tanh() + (v[1] / v[0]).exp()).log(),
}

xs = [1.3, 0.7]
for name, fn in tests.items():
    err = gradcheck(fn, xs)
    print(f"{name:16s} max rel error = {err:.2e}   {'OK' if err < 1e-6 else 'FAIL'}")

add              max rel error = 6.99e-11   OK
mul              max rel error = 6.49e-11   OK
sub              max rel error = 4.11e-11   OK
div              max rel error = 3.00e-11   OK
pow              max rel error = 3.87e-11   OK
exp              max rel error = 3.03e-11   OK
log              max rel error = 4.25e-11   OK
tanh             max rel error = 2.22e-12   OK
reuse: x*x + x   max rel error = 4.73e-11   OK
mixed            max rel error = 1.39e-10   OK


## Step 6: Two classic bugs

### 6a. Forgetting to reset gradients

Gradients use `+=`, so calling `backward()` twice **accumulates**. This is exactly why every training loop calls `zero_grad()` before each step.

In [23]:
x = Value(1.5)
y = x * x                     # dy/dx = 2x = 3.0

y.backward(); print("after 1st backward:", x.grad)
y.backward(); print("after 2nd backward:", x.grad, " <- doubled, nobody reset it")

after 1st backward: 3.0
after 2nd backward: 6.0  <- doubled, nobody reset it


### 6b. Using `=` instead of `+=`

If an op *overwrites* gradients instead of adding, everything still runs, nothing crashes, and the loss may even go down. But whenever a value is used twice (here $x \cdot x$), one path's gradient silently erases the other's.

This is the kind of bug the gradient check exists for.

In [24]:
class BuggyValue(Value):
    def __mul__(self, other):
        other = other if isinstance(other, Value) else BuggyValue(other)
        out = BuggyValue(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad  = other.data * out.grad    # BUG: '=' instead of '+='
            other.grad = self.data  * out.grad
        out._backward = _backward
        return out

x = BuggyValue(1.5)
y = x * x
y.backward()
print("buggy grad  :", x.grad)
print("correct grad:", 2 * 1.5)

# the gradient check catches it immediately
buggy_err = abs(x.grad - 3.0) / (abs(x.grad) + 3.0)
print(f"rel error = {buggy_err:.2e}  ->  {'OK' if buggy_err < 1e-6 else 'FAIL'}")

buggy grad  : 1.5
correct grad: 3.0
rel error = 3.33e-01  ->  FAIL


## Summary

- **Autograd = local derivatives + chain rule + the right order.** No op knows about the whole model.
- **Gradients add** when a value is used more than once (this is why it is `+=`).
- **Finite differences are the referee**: too slow for training, perfect for proving correctness.

## Next: Notebook 2, the same idea on NumPy arrays

The calculus stays the same; the new difficulty is **shapes**:
- `matmul`: the gradient of $XW$ w.r.t. $W$ is $X^\top \cdot \text{grad}$, and the shapes must line up.
- **broadcasting**: if a bias of shape `(4,)` was added to every row of a `(32, 4)` batch, its gradient must be **summed back** to `(4,)`.